# Module 8 lab · Training networks that learn

The code from every lesson of [this module](https://aicourses.dev/research/training), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 8.2 From micrograd to PyTorch: what .backward() does

PyTorch does what your micrograd engine did, for any size of model. You map each part of your engine onto it: which numbers track gradients, how gradients are reset between steps, and how to switch tracking off.

[Read the lesson](https://aicourses.dev/research/training/pytorch-autograd)

### Your engine, in PyTorch’s words

Here is the neuron from lesson 7.6 one more time, now in PyTorch:

In [ ]:
import torch

w = torch.tensor(0.5, requires_grad=True)   # a knob: track it
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(2.0)                       # data: no gradient needed

L = (w * x + b - 3) ** 2
print(L.item(), L.grad_fn.name())   # 1.0 PowBackward0
L.backward()
print(w.grad, b.grad)     # tensor(-4.) tensor(-2.)
print(x.grad)             # None

### Switch 2: gradients add up, so reset them

Run the forward and backward passes again without resetting:

In [ ]:
L = (w * x + b - 3) ** 2      # a fresh forward pass
L.backward()
print(w.grad, b.grad)         # tensor(-8.) tensor(-4.)

w.grad, b.grad = None, None   # reset, then try again
L = (w * x + b - 3) ** 2
L.backward()
print(w.grad, b.grad)         # tensor(-4.) tensor(-2.)

### Switch 3: turning tracking off

Updating a weight is not part of the model’s calculation, so it must not be recorded. Inside a `with torch.no_grad():` block, nothing is:

In [ ]:
with torch.no_grad():         # don't record what happens in here
    w -= 0.1 * w.grad         # one step of gradient descent
    b -= 0.1 * b.grad
print(w.item(), b.item())     # 0.8999999761581421 1.2000000476837158
print(((w * x + b - 3) ** 2).item())   # 0.0

### The same, for whole layers

None of this changes when the knobs are tensors. The gradient of a weight matrix is a matrix of the same shape, one slope per weight, and an optimizer does the update and the reset for every parameter at once:

In [ ]:
torch.manual_seed(0)
neuron = torch.nn.Linear(3, 1)              # 3 weights and a bias
x3 = torch.tensor([0.9, 0.6, 0.3])          # the pizza inputs
loss = ((neuron(x3) - 1.0) ** 2).sum()
loss.backward()
print(neuron.weight.grad.shape)             # torch.Size([1, 3])
print(neuron.weight.grad)    # tensor([[-2.4939, -1.6626, -0.8313]])

opt = torch.optim.SGD(neuron.parameters(), lr=0.1)
opt.step()                       # p -= lr * p.grad, for every p
opt.zero_grad()                             # reset every .grad to None
print(neuron.weight.grad)                   # None

## 8.3 Cross-entropy in practice: the stable form, and the loss at the start

Computers cannot hold very large numbers, so the softmax from Module 6 can overflow, and real code uses a rearranged, safe version. Also, an untrained model guessing evenly among N choices starts with a loss near the log of N. Checking that number is your first test.

[Read the lesson](https://aicourses.dev/research/training/softmax-and-cross-entropy)

### How big is too big

PyTorch stores numbers by default as **float32**: 32 bits each, which hold up to about $3.4 \times 10^{38}$. Since $\ln(3.4 \times 10^{38}) \approx 88.7$, any score above about 88.7 overflows when you raise $e$ to it:

In [ ]:
import torch

print(torch.exp(torch.tensor([88.0, 89.0])))
# tensor([1.6516e+38,        inf])

`1.6516e+38` is code’s way of writing $1.6516 \times 10^{38}$, and `inf` is how a computer writes “too big to hold”. One `inf` in a softmax ruins it, because `inf ÷ inf` has no answer and comes out as `nan`, short for “not a number”:

In [ ]:
s = torch.tensor([1000.0, 1001.0, 1002.0])
e = torch.exp(s)
print(e)              # tensor([inf, inf, inf])
print(e / e.sum())    # tensor([nan, nan, nan])

### The fix: subtract the biggest score

Work it for 1000, 1001 and 1002. Subtract 1002 to get −2, −1 and 0. Then $e^{-2} \approx 0.135$, $e^{-1} \approx 0.368$, $e^0 = 1$, the total is 1.503, and the probabilities are about **0.090, 0.245 and 0.665**, exactly the softmax of 0, 1 and 2.

In [ ]:
def softmax(s):
    e = torch.exp(s - s.max())     # the biggest score becomes 0
    return e / e.sum()

print(softmax(s))                  # tensor([0.0900, 0.2447, 0.6652])
print(torch.softmax(s, dim=0))     # tensor([0.0900, 0.2447, 0.6652])

### The other end: a probability that rounds to zero

Large scores overflow; large gaps cause the opposite problem. With scores 0 and 200, the first answer’s probability is $e^{-200}$, about $10^{-87}$, far below the smallest number float32 can hold, so it rounds to exactly 0. If that was the right answer, the cross-entropy is $-\ln 0$, which is infinite:

In [ ]:
s2 = torch.tensor([0.0, 200.0])
p = torch.softmax(s2, dim=0)
print(p, -torch.log(p[0]))         # tensor([0., 1.]) tensor(inf)

The second term is called **log-sum-exp**, and it is computed safely by the same trick: take out the biggest score $m$. Each $e^{s_j}$ is $e^m \times e^{s_j - m}$, and the log turns that $\times\, e^m$ into $+\, m$, so $\ln \sum_j e^{s_j} = m + \ln \sum_j e^{s_j - m}$. For scores 0 and 200, $m = 200$ and the sum inside is $e^{-200} + 1$, which is 1 to every digit float32 keeps, so log-sum-exp is 200. Then $\ln p_0 = 0 - 200 = -200$ and the loss is exactly 200: large, but a real number training can learn from.

In [ ]:
import torch.nn.functional as F

print(torch.log_softmax(s2, dim=0))    # tensor([-200.,    0.])
right = torch.tensor([0])              # the first answer was right
print(F.cross_entropy(s2.unsqueeze(0), right))   # tensor(200.)

### The loss at the start

The last column is GPT-2’s vocabulary of 50,257 tokens: an untrained language model of that size should start near 10.8. Check it on a fresh 10-class model:

In [ ]:
torch.manual_seed(0)
model = torch.nn.Linear(784, 10)            # an untrained classifier
x = torch.rand(256, 784)                    # 256 fake pictures
y = torch.randint(0, 10, (256,))            # random right answers
print(F.cross_entropy(model(x), y).item())  # 2.3610217571258545

with torch.no_grad():
    model.weight *= 10                      # a badly scaled start
print(F.cross_entropy(model(x), y).item())  # 5.3961052894592285

## 8.4 The training loop from a blank notebook

You write, from an empty notebook, the loop every later run reuses. It feeds data in small batches and checks progress on a validation set, kept apart from both training and test data. It saves snapshots called checkpoints and runs on a GPU.

[Read the lesson](https://aicourses.dev/research/training/training-loop)

### The data: three spirals

You need data that is quick to make and hard enough to need a network. Three spiral arms wound round each other, 200 points each, one class per arm, will do: no straight line can separate them.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)                    # the same run every time

def make_spirals(n=200, arms=3):
    """Three spiral arms of n points each, one class per arm."""
    xs, ys = [], []
    for arm in range(arms):
        r = torch.linspace(0.05, 1, n)            # out from the middle
        t = torch.linspace(arm * 4, arm * 4 + 4, n)   # round and round
        t = t + 0.2 * torch.randn(n)              # a little noise
        xs.append(torch.stack([r * t.sin(), r * t.cos()], dim=1))
        ys.append(torch.full((n,), arm))
    return torch.cat(xs), torch.cat(ys)

X, y = make_spirals()
print(X.shape, y.shape)   # torch.Size([600, 2]) torch.Size([600])

### Three slices: train, validation, test

The points were made arm by arm, so shuffle them before cutting the data into three slices:

In [ ]:
perm = torch.randperm(len(X))           # shuffle before splitting
X, y = X[perm], y[perm]
train = TensorDataset(X[:360], y[:360])         # 60%: learn from
val = TensorDataset(X[360:480], y[360:480])     # 20%: check progress
test = TensorDataset(X[480:], y[480:])          # 20%: the very end

loader = DataLoader(train, batch_size=32, shuffle=True)
xb, yb = next(iter(loader))             # peek at one batch
print(xb.shape, yb.shape)   # torch.Size([32, 2]) torch.Size([32])
print(len(loader))          # 12 batches per epoch

### The model, the device and a way to measure

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 3),                   # one score per spiral arm
).to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1)

def evaluate(dataset):
    model.eval()                        # switch to measuring mode
    with torch.no_grad():               # no graph needed
        xs, ys = dataset.tensors
        xs, ys = xs.to(device), ys.to(device)
        scores = model(xs)
        loss = F.cross_entropy(scores, ys).item()
        acc = (scores.argmax(dim=1) == ys).float().mean().item()
    model.train()                       # back to training mode
    return loss, acc

print(evaluate(val))   # (1.1061..., 0.333...): near ln 3 = 1.0986

### The loop

The heart of it is five lines: one that moves the batch to the device, then the same four moves, in the same order, as your micrograd loop in lesson 7.8:

In [ ]:
best = float("inf")
for epoch in range(1, 201):
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = F.cross_entropy(model(xb), yb)   # forward and loss
        opt.zero_grad()                         # reset gradients
        loss.backward()                         # backward pass
        opt.step()                              # one step downhill
    val_loss, val_acc = evaluate(val)
    if val_loss < best:                         # best so far: save it
        best = val_loss
        torch.save({"model": model.state_dict(),
                    "opt": opt.state_dict(),
                    "epoch": epoch}, "best.pt")
    if epoch % 40 == 0:
        print(epoch, round(loss.item(), 3),
              round(val_loss, 3), round(val_acc, 3))

*The lesson shows:*

```text
40 0.208 0.391 0.8
80 0.038 0.094 0.983
120 0.015 0.03 1.0
160 0.018 0.018 1.0
200 0.045 0.015 1.0
```

### Test once, at the end

Load the best checkpoint and score the test set, once:

In [ ]:
ckpt = torch.load("best.pt")
model.load_state_dict(ckpt["model"])    # back to the best weights
print(ckpt["epoch"], evaluate(test))    # the test set, used once

## 8.5 Optimizers: smarter ways to take each downhill step

An optimizer is the rule that turns gradients into weight changes. Plain gradient descent, momentum and AdamW are three such rules, and language models use AdamW.

[Read the lesson](https://aicourses.dev/research/training/optimizers-and-schedules)

### The race

In [ ]:
import torch

def loss_fn(w):
    return 0.5 * w[0] ** 2 + 50 * w[1] ** 2   # long and narrow

def race(make_opt, max_steps=2000):
    """Steps until the loss drops below 0.001 (None: never)."""
    w = torch.tensor([-5.0, 1.0], requires_grad=True)
    opt = make_opt([w])
    for step in range(1, max_steps + 1):
        opt.zero_grad()
        loss_fn(w).backward()
        opt.step()
        if loss_fn(w).item() < 0.001:
            return step
    return None

SGD, AdamW = torch.optim.SGD, torch.optim.AdamW
print(race(lambda p: SGD(p, lr=0.01)))                  # 470
print(race(lambda p: SGD(p, lr=0.019)))                 # 246
print(race(lambda p: SGD(p, lr=0.021)))                 # None
print(race(lambda p: SGD(p, lr=0.01, momentum=0.9)))    # 94
print(race(lambda p: AdamW(p, lr=0.1, weight_decay=0))) # 82

## 8.6 Learning-rate schedules: warmup, decay and clipping

The best step size changes during a run. Warmup starts small, decay shrinks the step near the end, and clipping caps any step that is too big. The learning rate is the first thing to tune.

[Read the lesson](https://aicourses.dev/research/training/learning-rate-schedules)

### Decay: slow down near the end

$\eta_{\text{peak}}$ is the peak learning rate, $p$ is the progress through the decay, from 0 at its start to 1 at the end, and $f$ is the floor as a fraction of the peak. Here $\pi$ stands for half a turn (180°), so $\cos(\pi p)$ slides from 1 down to −1 and the cosine part, $\tfrac{1}{2}(1 + \cos(\pi p))$, runs smoothly from 1 down to 0. With $f = 0.1$: at $p = 0$ the rate is the peak; at $p = 0.5$, $\cos(\pi p) = 0$, so the cosine part is 0.5 and the rate is $0.1 + 0.9 \times 0.5 = 0.55$ of the peak; and at $p = 1$ it is a tenth of the peak. GPT-3 and LLaMA both decayed to 10% of their peak this way.

In [ ]:
import math

def lr_at(step, peak=0.03, warmup=36, total=360, floor=0.1):
    """Linear warmup, then cosine decay down to floor × peak."""
    if step < warmup:
        return peak * (step + 1) / warmup     # climb from near 0
    progress = (step - warmup) / (total - warmup)   # 0 to 1
    cosine = 0.5 * (1 + math.cos(math.pi * progress))   # 1 to 0
    return peak * (floor + (1 - floor) * cosine)

for step in [0, 17, 35, 198, 359]:
    print(step, round(lr_at(step), 5))
# 0 0.00083
# 17 0.015
# 35 0.03
# 198 0.0165
# 359 0.003

### Clipping: a speed limiter for steps

Now and then one batch produces a huge gradient, and one huge step can undo hours of training. **Gradient clipping** caps it. Treat every gradient in the model as one long vector and measure its length, the **global gradient norm** (the vector length from Module 5). If it is longer than a limit $c$, scale every gradient down by $c$ divided by the norm, so the direction stays the same and only the length is capped. A gradient of $(3, 4)$ has length 5; clipped to 1, it becomes $(0.6, 0.8)$.

In [ ]:
import torch

w = torch.zeros(2, requires_grad=True)
w.grad = torch.tensor([3.0, 4.0])          # a gradient of length 5
norm = torch.nn.utils.clip_grad_norm_([w], max_norm=1.0)
print(norm, w.grad)    # tensor(5.) tensor([0.6000, 0.8000])

w.grad = torch.tensor([0.3, 0.4])          # length 0.5: left alone
norm = torch.nn.utils.clip_grad_norm_([w], max_norm=1.0)
print(norm, w.grad)    # tensor(0.5000) tensor([0.3000, 0.4000])

## 8.7 Overfitting, regularization and generalization

A model can memorise its training examples and still fail on new ones. That is overfitting. You catch it with a validation set, and prevent it with weight decay and early stopping, while keeping test data out of training.

[Read the lesson](https://aicourses.dev/research/training/generalization)

### Fitting too well

Take 10 noisy points from a smooth wave, and 100 fresh points from the same wave to check on. Fit a straight line (degree 1), a cubic (degree 3) and a degree-9 polynomial, which has 10 knobs, one per point. `np.polyfit` finds the best curve of a given degree; `np.polyval` evaluates it.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make(n):                              # a smooth curve plus noise
    x = rng.uniform(0, 1, n)
    return x, np.sin(2 * np.pi * x) + rng.normal(0, 0.2, n)

x_tr, y_tr = make(10)                     # 10 points to learn from
x_va, y_va = make(100)                    # 100 new points to check

def mse(coefs, x, y):
    return np.mean((np.polyval(coefs, x) - y) ** 2)

for degree in [1, 3, 9]:
    coefs = np.polyfit(x_tr, y_tr, degree)   # best fit of this degree
    print(degree, round(mse(coefs, x_tr, y_tr), 4),
          round(mse(coefs, x_va, y_va), 4))
# 1 0.2472 0.3791
# 3 0.0238 0.0527
# 9 0.0 0.4073

### Catching it: watch the validation curve

With a network you watch instead. Here is one far bigger than its data needs, two hidden layers of 128 neurons, trained on 40 points from two overlapping clouds and checked on 1,000 more. `make_clouds` shifts class 1’s points 1.5 along both axes.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(0)
def make_clouds(n):                 # two overlapping clouds of points
    y = torch.randint(0, 2, (n,))
    x = torch.randn(n, 2) + 1.5 * y.unsqueeze(1).float()
    return x, y

pts_tr, lab_tr = make_clouds(40)    # a small training set
pts_va, lab_va = make_clouds(1000)  # a big validation set
torch.manual_seed(1)
model = nn.Sequential(nn.Linear(2, 128), nn.ReLU(),
                      nn.Linear(128, 128), nn.ReLU(),
                      nn.Linear(128, 2))
opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0)

In [ ]:
def accuracy(x, y):
    return (model(x).argmax(dim=1) == y).float().mean().item()

best = (float("inf"), 0)
for epoch in range(1, 2001):
    loss = F.cross_entropy(model(pts_tr), lab_tr)  # all 40 at once
    opt.zero_grad()
    loss.backward()
    opt.step()
    with torch.no_grad():
        val = F.cross_entropy(model(pts_va), lab_va).item()
        best = min(best, (val, epoch))           # lowest val so far
        if epoch in (33, 2000):
            print(epoch, round(loss.item(), 3), round(val, 3),
                  round(accuracy(pts_tr, lab_tr), 3),
                  round(accuracy(pts_va, lab_va), 3))
print(best)

*The lesson shows:*

```text
33 0.406 0.379 0.8 0.828
2000 0.001 8.916 1.0 0.701
(0.37931525707244873, 33)
```

### Preventing it: weight decay

$L$ is the usual loss and $L_{\text{total}}$ the one training now makes small; $\lambda$ sets how strongly large weights are punished, and the sum runs over every weight $w_i$. With weights 3 and −2 and $\lambda = 0.01$, the penalty is $0.01 \times (9 + 4) = 0.13$. For plain gradient descent, this penalty is exactly weight decay from lesson 8.5, at a rate of $2\lambda$: its slope, $2\lambda w$, shrinks every weight a little each step. For the polynomial there is a formula for the best weights with the penalty included, so you can see its effect directly. `np.vander` builds the table of $x^9, x^8, \dots, 1$ for each point:

In [ ]:
A_tr = np.vander(x_tr, 10)          # columns x⁹, x⁸, ..., x, 1
A_va = np.vander(x_va, 10)
for lam in [0, 1e-4, 1]:
    # the weights that minimise squared error + lam × sum of w²
    w = np.linalg.solve(A_tr.T @ A_tr + lam * np.eye(10),
                        A_tr.T @ y_tr)
    print(lam, round(np.mean((A_va @ w - y_va) ** 2), 4),
          round(np.abs(w).max(), 1))
# 0 0.4077 73024.4
# 0.0001 0.0654 14.2
# 1 0.3961 0.4

## 8.8 Measuring a classifier: accuracy, precision, recall and the confusion matrix

Accuracy hides mistakes when classes are uneven. Precision, recall and the confusion matrix show which mistakes a model makes, the first step towards Module 12’s evals.

[Read the lesson](https://aicourses.dev/research/training/measuring-classifiers)

### The threshold trade-off

A real classifier gives a score, and you choose the **threshold**: the score above which you say yes. Here is a simulated filter whose scores for spam are mostly higher than for real email, with some overlap, tried at three thresholds:

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

is_spam = np.zeros(1000, dtype=bool)
is_spam[:50] = True                     # 50 of 1,000 emails are spam
# a filter’s spam score: higher for spam, but the two overlap
score = np.where(is_spam, rng.normal(2.5, 1, 1000),
                 rng.normal(0, 1, 1000))

def report(threshold):
    flagged = score > threshold         # what the filter calls spam
    tp = np.sum(flagged & is_spam)      # spam, caught
    fp = np.sum(flagged & ~is_spam)     # real email, wrongly flagged
    fn = np.sum(~flagged & is_spam)     # spam that slipped through
    tn = np.sum(~flagged & ~is_spam)    # real email, left alone
    print(threshold, tp, fp, fn, tn, "accuracy", (tp + tn) / 1000,
          "precision", round(tp / (tp + fp), 3),
          "recall", round(tp / (tp + fn), 3))

for threshold in [1.0, 2.0, 3.0]:
    report(threshold)
print("never spam:", np.mean(~is_spam))   # accuracy of doing nothing

*The lesson shows:*

```text
1.0 49 152 1 798 accuracy 0.847 precision 0.244 recall 0.98
2.0 36 22 14 928 accuracy 0.964 precision 0.621 recall 0.72
3.0 16 0 34 950 accuracy 0.966 precision 1.0 recall 0.32
never spam: 0.95
```

### More than two classes

With more classes the confusion matrix grows a row and a column per class, and it shows exactly which classes get mixed up. Here are 12 made-up predictions for three kinds of clothing:

In [ ]:
names = ["T-shirt", "shirt", "sneaker"]
true = [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]   # 12 made-up pictures
pred = [0, 0, 0, 1, 0, 1, 1, 0, 2, 2, 2, 2]   # what a model said

cm = np.zeros((3, 3), dtype=int)
for t, p in zip(true, pred):
    cm[t, p] += 1                       # row: truth, column: guess
print(cm)
recall = cm.diagonal() / cm.sum(axis=1)       # per row
precision = cm.diagonal() / cm.sum(axis=0)    # per column
for name, p, r in zip(names, precision, recall):
    print(f"{name:8} precision {p:.2f}  recall {r:.2f}")

*The lesson shows:*

```text
[[3 1 0]
 [2 2 0]
 [0 0 4]]
T-shirt  precision 0.60  recall 0.75
shirt    precision 0.67  recall 0.50
sneaker  precision 1.00  recall 1.00
```

## 8.9 Why training breaks: initialization, activations and normalization

A network’s starting weights, called its initialization, can make the numbers inside it blow up or fade away until it stops learning. You train a badly started deep network and watch each layer’s outputs. Then you see why normalization, which rescales them, fixes it.

[Read the lesson](https://aicourses.dev/research/training/why-training-breaks)

### A deep network, three ways to start it

Here is a network of 10 ReLU layers, 256 neurons wide, for the three-arm spirals of lesson 8.4. Its weights are drawn at random with a spread you choose: `scale` divided by the square root of the number of inputs.

In [ ]:
import math
import torch
import torch.nn.functional as F
from torch import nn

def deep_net(scale, depth=10, width=256, norm=False):
    """10 ReLU layers whose weights have spread scale / √(inputs)."""
    layers, n_in = [], 2
    for _ in range(depth):
        lin = nn.Linear(n_in, width)
        nn.init.normal_(lin.weight, 0, scale / math.sqrt(n_in))
        nn.init.zeros_(lin.bias)
        layers.append(lin)
        if norm:
            layers.append(nn.LayerNorm(width))   # lesson’s last part
        layers.append(nn.ReLU())
        n_in = width
    layers.append(nn.Linear(width, 3))
    return nn.Sequential(*layers)

`nn.init.normal_` refills a layer’s weights with random numbers centred on 0 with the spread you give, and `nn.init.zeros_` sets its biases to 0. `*layers` hands the list’s items to `nn.Sequential` one by one. To watch it, measure the spread, the standard deviation from Module 3, of each layer’s outputs:

In [ ]:
def layer_spreads(model, x):
    """The standard deviation of every ReLU layer’s outputs."""
    spreads = []
    for layer in model:
        x = layer(x)
        if isinstance(layer, nn.ReLU):
            spreads.append(round(x.std().item(), 3))
    return spreads

Now start it three ways and look inside before any training:

In [ ]:
torch.manual_seed(0)
X, y = make_spirals()                 # the data from lesson 8.4
perm = torch.randperm(len(X))         # shuffled, as before
X, y = X[perm], y[perm]
for scale in [0.5, math.sqrt(2), 3.0]:
    torch.manual_seed(1)
    print(round(scale, 2), layer_spreads(deep_net(scale), X))

*The lesson shows:*

```text
0.5 [0.138, 0.049, 0.017, 0.006, 0.002, 0.001, 0.0, 0.0, 0.0, 0.0]
1.41 [0.391, 0.395, 0.388, 0.406, 0.391, 0.364, 0.4, 0.369, 0.36, 0.333]
3.0 [0.83, 1.777, 3.703, 8.231, 16.776, 33.15, 77.274, 151.139, 312.873, 613.601]
```

### What happens when you train them

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train(model, epochs=20):
    """Train on 360 points; return validation loss, first gradient."""
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    loader = DataLoader(TensorDataset(X[:360], y[:360]),
                        batch_size=32, shuffle=True)
    first_grads = []
    for epoch in range(epochs):
        for xb, yb in loader:
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            first_grads.append(model[0].weight.grad.norm().item())
            opt.step()
    with torch.no_grad():
        val = F.cross_entropy(model(X[360:480]), y[360:480]).item()
    return round(val, 3), f"{first_grads[0]:.1e}"

for scale in [0.5, math.sqrt(2), 3.0]:
    torch.manual_seed(1)
    print(round(scale, 2), train(deep_net(scale)))

*The lesson shows:*

```text
0.5 (1.107, '3.1e-06')
1.41 (0.004, '3.3e-02')
3.0 (nan, '5.8e+01')
```

### Normalization: rescale as you go

Instead of trusting the start, you can rescale each layer’s outputs on the way through. **Layer normalization**, or **LayerNorm**, published by Jimmy Ba, Jamie Kiros and Geoffrey Hinton in 2016, takes one example’s vector of outputs, subtracts their mean and divides by their spread. For $[2, 4, 6]$: the mean is 4, the gaps from it are −2, 0 and 2, so the spread is $\sqrt{(4 + 0 + 4)/3} = \sqrt{8/3} \approx 1.633$, and dividing the gaps by it gives about $[-1.22, 0, 1.22]$. Then it multiplies by a learned scale and adds a learned shift, which start at 1 and 0, so the network can undo the rescaling where that helps. The `norm=True` option in `deep_net` puts one after every linear layer:

In [ ]:
for scale in [0.5, 3.0]:
    torch.manual_seed(1)
    model = deep_net(scale, norm=True)
    print(scale, layer_spreads(model, X), train(model))

*The lesson shows:*

```text
0.5 [0.587, 0.604, 0.578, 0.562, 0.582, 0.576, 0.591, 0.574, 0.584, 0.571] (0.097, '1.9e-01')
3.0 [0.588, 0.604, 0.578, 0.562, 0.582, 0.576, 0.591, 0.574, 0.584, 0.571] (0.033, '3.1e-02')
```

## 8.10 Debugging a run that doesn’t learn

Training often fails without any error: the code runs and the loss misbehaves. Reading the curves to find the cause is a skill of its own, and some research-engineering interviews test it by handing you a loop that runs but does not learn.

[Read the lesson](https://aicourses.dev/research/training/debugging-training)

### Three checks before you train

**2. Overfit one batch.** A healthy model and loss can memorise a single batch of 32 examples until the loss is nearly 0. If yours cannot, the bug is in the model or the loss, not in the amount of data or training. The spiral data is the same as in lesson 8.4:

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

def small_net():
    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                         nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, 3))

def overfit_one_batch(loss_fn, steps=500):
    """Train on a single batch of 32; a healthy setup gets near 0."""
    torch.manual_seed(1)
    model = small_net()
    opt = torch.optim.AdamW(model.parameters(), lr=0.01)
    xb, yb = X[:32], y[:32]
    for _ in range(steps):
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return round(loss.item(), 4)
def double_softmax(scores, y):          # the bug: softmax first
    return F.cross_entropy(torch.softmax(scores, dim=1), y)
print(overfit_one_batch(F.cross_entropy))   # 0.0001
print(overfit_one_batch(double_softmax))    # 0.6765

**3. Check shapes, and check the metric too.** Broadcasting from Module 5 lets tensors of different shapes combine silently, and it can change a result without any error. Here a model gets every validation point right, and a small slip in the accuracy line hides it:

In [ ]:
torch.manual_seed(1)
model = small_net()
opt = torch.optim.AdamW(model.parameters(), lr=0.01)
for step in range(300):                 # all 360 points at once
    loss = F.cross_entropy(model(X[:360]), y[:360])
    opt.zero_grad()
    loss.backward()
    opt.step()

with torch.no_grad():
    guess = model(X[360:480]).argmax(dim=1)     # shape (120,)
right = y[360:480]                              # shape (120,)
print((guess == right).float().mean())          # the true accuracy
wrong = guess == right.unsqueeze(1)     # (120,) against (120, 1)
print(wrong.shape, wrong.float().mean())        # the bug

*The lesson shows:*

```text
tensor(1.)
torch.Size([120, 120]) tensor(0.3368)
```

## 8.11 Project: a classifier and a debugging logbook

Shows both halves of the job: training a model that works, and explaining why a broken one won’t learn. Your model must beat a baseline, the simplest model worth comparing against. Here that is one layer followed by softmax.

[Read the lesson](https://aicourses.dev/research/training/project-classifier-and-logbook)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

1. **Load and split the data.** Open a new Colab notebook and switch to a GPU runtime: Runtime > Change runtime type. If no GPU is free, the CPU runtime works too, only slower. You need torchvision, PyTorch’s companion library for images; Colab usually has it, and `!pip install torchvision` in a cell adds it if not. It downloads FashionMNIST into the runtime the first time.

The lesson installs this first:

In [ ]:
!pip install torchvision

In [ ]:
import math
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

torch.manual_seed(0)
to_tensor = transforms.ToTensor()       # pixels become 0 to 1
full = datasets.FashionMNIST("data", train=True, download=True,
                             transform=to_tensor)
test = datasets.FashionMNIST("data", train=False, download=True,
                             transform=to_tensor)
split = torch.Generator().manual_seed(0)    # the same split every run
train, val = random_split(full, [50_000, 10_000], generator=split)

loader = DataLoader(train, batch_size=128, shuffle=True)
xb, yb = next(iter(loader))
print(xb.shape, yb.shape, len(train), len(val), len(test))

2. **Look before you train.** Plot a dozen photos with their labels with matplotlib, and check every label from 0 to 9 appears in the training set. Then build both models and check their starting loss:

In [ ]:
def baseline():                         # softmax regression
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

def my_model():                         # yours: change freely
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 10))

for make in [baseline, my_model]:
    torch.manual_seed(0)
    model = make()
    loss = F.cross_entropy(model(xb), yb).item()
    print(make.__name__, round(loss, 3))   # compare with ln 10

*The lesson shows:*

```text
Bug: softmax before F.cross_entropy           Seed: 0
Symptom: what you saw: the first loss, the shape of the loss
         and gradient-norm curves, any floor, the accuracy
Diagnosis: which check or curve pointed to the cause, and why
Fix: the one line you changed
After the fix: the same numbers again, back to normal
```